# Course 5: Building a Small Bangla Language Model
## Session 12: Release and write-up

**Goal:** package the model so a stranger can load it, and publish it.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network. The upload and demo cells were not run.

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

In [ ]:
# A tiny model and tokenizer trained on the built-in sample, so this notebook runs anywhere.
# Replace with your own tokenizer file and checkpoint from sessions 4 and 6.
torch.manual_seed(0)
tok = train_bpe(SAMPLE_LINES * 20, 300)
EOT = tok.token_to_id("<|endoftext|>")
ids = []
for line in SAMPLE_LINES * 20:
    ids += tok.encode(line).ids + [EOT]
data = torch.tensor(ids)
cfg = Config(vocab_size=tok.get_vocab_size(), block_size=32, n_layer=2, n_head=2, d_model=64)
model = GPT(cfg)

def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):
    x, y = get_batch(data, cfg.block_size, 32)
    _, loss = model(x, y)
    opt.zero_grad(); loss.backward(); opt.step()
print("tiny model trained, final loss", round(loss.item(), 3))

### 1. Save the release files

In [ ]:
import json, os
os.makedirs("release", exist_ok=True)
torch.save(model.state_dict(), "release/model.pt")
tok.save("release/tokenizer.json")
with open("release/config.json", "w") as f:
    json.dump(cfg.__dict__, f, indent=1)
with open("release/README.md", "w", encoding="utf-8") as f:
    f.write("# My small Bangla model\n\nTODO: paste your model card here.\n")
print(sorted(os.listdir("release")))

### 2. Load from the files alone
This is what a stranger does. Put the model code in a file of its own in your repository.

In [ ]:
from tokenizers import Tokenizer
with open("release/config.json") as f:
    cfg2 = Config(**json.load(f))
model2 = GPT(cfg2)
model2.load_state_dict(torch.load("release/model.pt", map_location="cpu"))
model2.eval()
tok2 = Tokenizer.from_file("release/tokenizer.json")
torch.manual_seed(0)
out = model2.generate(torch.tensor([tok2.encode("আমি").ids]), 12, temperature=0.7)
print(tok2.decode(out[0].tolist()))

### 3. Check that the reload matches

In [ ]:
x = data[:cfg.block_size].unsqueeze(0)
model.eval()
with torch.no_grad():
    print("identical outputs:", torch.allclose(model(x)[0], model2(x)[0]))

### 4. Upload to the Hugging Face Hub (TODO, NOT RUN)
Needs an account and a token.

In [ ]:
# !pip install huggingface_hub
# from huggingface_hub import HfApi, login
# login()                                             # paste a write token
# api = HfApi()
# api.create_repo("YOUR-USERNAME/small-bangla-gpt", exist_ok=True)
# api.upload_folder(folder_path="release", repo_id="YOUR-USERNAME/small-bangla-gpt")

### 5. A small demo (TODO, NOT RUN)

In [ ]:
# !pip install gradio
# import gradio as gr
# def complete(prompt, temperature=0.8):
#     out = model2.generate(torch.tensor([tok2.encode(prompt).ids]), 60, temperature=temperature)
#     return tok2.decode(out[0].tolist())
# gr.Interface(complete, [gr.Textbox(label="শুরু করুন"), gr.Slider(0.1, 1.5, 0.8)], "text").launch()

### Report checklist (TODO)
- Goal and design brief
- Data, with the data card
- Tokenizer and model
- Training, with curves
- Evaluation: perplexity, human ratings, downstream task, instruction comparison, compression
- Limits
- What I would do next
- A classmate reproduced one number from the public release